# 00 - Setup

Team UL, DIRISA Student Datathon Challenge 2026: The Democratic Funnel.

Run this first. It works in Google Colab and on a normal computer.

The code (`src/`, `notebooks/`, `app/`) is on GitHub. The data, models and figures are in the shared Drive folder
`DIRISA_SDC` (or in the repo's `data/` folder when you work locally).

This notebook:
1. finds the repo (in Colab it clones or updates it) and connects Google Drive
2. creates the project folders
3. installs any missing packages (Colab only)
4. loads the shared code and settings
5. records the package versions
6. lists the raw data files and checks the sources list

First time in Colab: in Drive, right-click `DIRISA_SDC`, choose Organise, then Add shortcut to My Drive.
The repo is private, so add your GitHub token in Colab under Secrets, with the name `GITHUB_TOKEN`.

Do not save work only in `/content/`. Colab deletes it when the session ends.

## 1. Locate the repo and mount Drive

In [ ]:
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Thato20-M/democratic-funnel.git"   # <- team repo URL
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/democratic-funnel")
    url = REPO_URL
    try:                                    # private repo: token from Colab Secrets
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass                                # public repo, or no token set
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", url, str(REPO_DIR)], check=True)
else:
    # local: walk up from the notebook until we find src/config.py
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "src" / "config.py").exists())

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

## 2. Extra packages (Colab only)
Colab already has pandas, numpy, scikit-learn, statsmodels, lightgbm, geopandas, matplotlib and seaborn. Locally, use `pip install -r requirements.txt` once instead.

In [ ]:
if IN_COLAB:
    %pip install -q shap pdfplumber

## 3. Shared code, folders and settings
`autoreload` means changes to `src/` are picked up without restarting the runtime.

In [ ]:
from src import config
from src.config import RAW, INTERIM, PROCESSED, MODELS, FIGURES, SEED
from src.io_utils import read_any, clean_columns, inventory
from src.privacy import drop_personal

if config.IN_COLAB and not config.DRIVE_PROJECT.exists():
    raise FileNotFoundError(
        "DIRISA_SDC not found in My Drive. Right-click the shared folder -> "
        "Organise -> Add shortcut -> My Drive, then re-run.")

config.make_dirs()
config.set_seed()
print(config.describe())

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 110
sns.set_theme(style="whitegrid")

## 4. Log the environment
Saved to `environment.json` in the work root so results can be reproduced with the same versions.

In [ ]:
import platform, datetime
import importlib.metadata as md

PKGS = ["pandas", "numpy", "scikit-learn", "statsmodels", "lightgbm", "shap",
        "geopandas", "matplotlib", "seaborn", "openpyxl", "streamlit"]

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return "not installed"

env = {"logged": datetime.datetime.now().isoformat(timespec="seconds"),
       "environment": "colab" if IN_COLAB else "local",
       "python": platform.python_version(),
       "packages": {p: _ver(p) for p in PKGS}}
(config.WORK_ROOT / "environment.json").write_text(json.dumps(env, indent=2))
env

## 5. Raw-data inventory
One row per CSV file / Excel sheet, with the detected header row (IEC reports often have title rows above the real header). Re-run whenever new files land in `data/raw`.

In [ ]:
inv = inventory()
inv.to_csv(config.WORK_ROOT / "data_inventory.csv", index=False)
print(f"{inv['file'].nunique() if len(inv) else 0} files, {len(inv)} sheets/tables")
inv

## 6. Check the sources list
`docs/sources.csv` is in the repo and is part of the submission. Every source needs a link and the date we
downloaded it. The registration dashboard also needs the date of the snapshot.

In [ ]:
sources = pd.read_csv(config.SOURCES_LOG)
todo = sources[sources["url"].isna() | sources["accessed_date"].isna()]
print(f"{len(todo)} of {len(sources)} sources still need a URL and/or access date")
todo[["source_id", "url", "accessed_date", "notes"]]

## How we work as a team
- One notebook per step, and each notebook has one owner.
- Every notebook starts with the setup cells from this notebook.
- Functions we use in more than one notebook go in `src/`.
- Read from `data/raw` or `data/interim`, and write to `data/interim` or `data/processed`. Never change the raw files.
- Run `clean_columns()` straight after reading a file.
- Join tables on the municipality code, `muni_code` (for example `LIM354` or `TSH`). District municipalities are
  not rows; we keep `district_code` as a column.
- Add every download to `docs/sources.csv` (link, date downloaded, snapshot date).
- In Colab the last save wins, so two people should not edit the same notebook at the same time. Save your
  notebook to GitHub every day (File, Save a copy in GitHub).
- Candidate lists contain personal data: run `drop_personal()` straight away, keep only counts, and never
  share or commit them.